# 01 — Coleta e integração
Fonte: portal oficial de [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf), arquivos **agrupados por ocorrência**. Os links são obtidos da página oficial para evitar URLs desatualizadas.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile, io, re
import gdown

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
for p in [DATA/"raw", DATA/"processed", OUT/"auditoria", OUT/"eda", OUT/"estatistica", OUT/"modelos", OUT/"interpretabilidade", OUT/"modelo_final"]:
    p.mkdir(parents=True,exist_ok=True)

# IDs conferidos nos links da página oficial da PRF (arquivos agrupados por ocorrência).
# https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf
ids = {
    2020: "1esu6IiH5TVTxFoedv6DBGDd01Gvi8785",
    2021: "12xH8LX9aN2gObR766YN3cMcuycwyCJDz",
    2022: "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-",
    2023: "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq",
    2024: "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO",
    2025: "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR",
}


In [ ]:
bases=[]
for ano, file_id in sorted(ids.items()):
    pasta = DATA/"raw"/str(ano)
    pasta.mkdir(parents=True,exist_ok=True)
    arquivo = pasta/f"datatran{ano}.zip"
    if not arquivo.exists():
        print(f"Baixando {ano} da PRF...")
        gdown.download(f"https://drive.google.com/uc?id={file_id}", str(arquivo), quiet=False)
    if not zipfile.is_zipfile(arquivo):
        raise RuntimeError(f"Arquivo PRF {ano} não está em ZIP válido: {arquivo}. Confirme se o link oficial ainda é o mesmo.")
    with zipfile.ZipFile(arquivo) as z:
        csvs=[n for n in z.namelist() if n.lower().endswith(".csv")]
        assert csvs,f"Sem CSV para {ano}"
        conteudo=z.read(csvs[0])
    df_ano=None
    for enc in ("utf-8-sig","latin-1"):
        try:
            df_ano=pd.read_csv(io.BytesIO(conteudo),sep=";",encoding=enc,low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    assert df_ano is not None, f"Falha na leitura do CSV {ano}"
    df_ano.columns=df_ano.columns.str.strip().str.lower()
    df_ano["ano_fonte"]=ano
    bases.append(df_ano)
    print(f"{ano}: {len(df_ano):,} registros")

df=pd.concat(bases,ignore_index=True,sort=False)
for c in ("km","latitude","longitude"):
    df[c]=pd.to_numeric(df[c].astype("string").str.replace(",",".",regex=False),errors="coerce")

arquivo_saida=DATA/"processed"/"prf_acidentes_2020_2025.parquet"
df.to_parquet(arquivo_saida,index=False)
print("Consolidado:",df.shape,"IDs repetidos:",df.id.duplicated().sum())
print("Arquivo:",arquivo_saida)
